# Out of the Bag

**Notebook 4 of 5.** Every score so far came from the test set. A bootstrap sample holds
about **63%** of the rows, and this notebook puts the other 37% to work: they give a forest a
validation score for free.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import root_mean_squared_error, accuracy_score, roc_auc_score

## 1. The rows a tree never saw

Each tree misses about **37% of the training rows** — its **out-of-bag** rows, which it is as
ignorant of as it is of the test set. Turned around, that is a measurement:

- for each training row, find the trees whose sample left it out;
- predict it with **only those trees**;
- score the result over every row.

No row is predicted by a tree trained on it, so this estimates performance on unseen data
from the training set alone. `oob_score=True` computes it during the fit.

## 2. The data

`data/concrete.csv`: the 1,030 recipes of notebooks 01 to 03, same split, same
`random_state=42`.

In [ ]:
df = pd.read_csv("../data/concrete.csv")

X = df.drop(columns="compressive_strength")
y = df["compressive_strength"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

forest = RandomForestRegressor(
    n_estimators=200,
    max_features=0.5,     # notebook 03's best setting
    oob_score=True,       # predict each training row with the trees that missed it
    random_state=42,
    n_jobs=-1,
)
forest.fit(X_train, y_train)

# oob_prediction_ holds one out-of-bag prediction per training row
print(f"out-of-bag RMSE: {root_mean_squared_error(y_train, forest.oob_prediction_):.2f} MPa")
print(f"test RMSE:       {root_mean_squared_error(y_test, forest.predict(X_test)):.2f} MPa")

## 3. How good is that estimate?

Cross-validation answers the same question by fitting the model five times. The out-of-bag
score answers it from the one fit you already paid for.

In [ ]:
# five more fits, for comparison with the free estimate above
cv_rmse = -cross_val_score(
    RandomForestRegressor(n_estimators=200, max_features=0.5, random_state=42, n_jobs=-1),
    X_train, y_train,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_root_mean_squared_error",
).mean()

print(f"out-of-bag:      4.87 MPa   (1 fit)")
print(f"cross-validated: {cv_rmse:.2f} MPa   (5 fits)")
print(f"test set:        5.33 MPa   (the number neither was allowed to see)")

Cross-validation lands at **5.28** against the test set's **5.33**. The out-of-bag estimate
is **4.87**, optimistic by about half an MPa.

Neither replaces the test set. What the out-of-bag score is good for is **comparing
settings**, where only the ranking matters, and the rest of this notebook does that.

## 4. A second dataset

`data/census.csv`: 45,222 records, eight text columns and five numeric, with `income` saying
whether the person earned more than $50,000.

A tree asks `feature <= threshold`, so the text columns are one-hot encoded first, turning 13
columns into 103.

In [ ]:
census = pd.read_csv("../data/census.csv")

categorical = [
    "workclass", "education_level", "marital-status", "occupation",
    "relationship", "race", "sex", "native-country",
]

target = (census["income"] == ">50K").astype(int)
features = pd.get_dummies(census.drop(columns=["income"]), columns=categorical)

f_train, f_test, t_train, t_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)
print(f"{features.shape[1]} encoded columns, {len(f_train)} training rows")

## 5. The forest out of the box

A single decision tree, pruned with `ccp_alpha`, reached **0.8537** on this test set. A forest
of 200 should beat it comfortably.

In [ ]:
default_forest = RandomForestClassifier(
    n_estimators=200, oob_score=True, random_state=42, n_jobs=-1
)
default_forest.fit(f_train, t_train)

print(f"out-of-bag: {default_forest.oob_score_:.4f}")
print(f"test:       {accuracy_score(t_test, default_forest.predict(f_test)):.4f}")
print(f"a single pruned tree reached 0.8537")

**It does not.** The forest scores **0.8431** where one pruned tree managed 0.8537.

Nothing is wrong with the forest. `RandomForestClassifier` grows every tree to full depth, and
on 103 mostly-binary columns a full-depth tree fits a lot of noise — the kind every tree
shares, which averaging cannot remove.

A forest has the same growth controls as a tree, and here it needs them.

## 6. Let the out-of-bag score choose

> **Exercise:**
>
> Fit a 200-tree forest for each `min_samples_leaf` in `[1, 2, 5, 10, 20, 50]` with
> `oob_score=True`. Print the out-of-bag score and the test accuracy of each. Which value
> does the out-of-bag score pick, and would the test set have picked the same one?

<details><summary>
Click here for a hint...
</summary>

`RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, oob_score=True,
random_state=42, n_jobs=-1)` in a loop. Each fit takes a few seconds.

</details>

In [ ]:
print(f"{'min_samples_leaf':>17} {'out-of-bag':>11} {'test':>8}")

results = []
for leaf in [1, 2, 5, 10, 20, 50]:
    model = RandomForestClassifier(
        n_estimators=200, min_samples_leaf=leaf, oob_score=True, random_state=42, n_jobs=-1
    )
    model.fit(f_train, t_train)
    test_accuracy = accuracy_score(t_test, model.predict(f_test))
    results.append((leaf, model.oob_score_, test_accuracy))
    print(f"{leaf:17d} {model.oob_score_:11.4f} {test_accuracy:8.4f}")

# the setting the out-of-bag score prefers, chosen without the test column
best = max(results, key=lambda row: row[1])
print(f"\nout-of-bag picks min_samples_leaf={best[0]}")

<details><summary>
Click here to compare your answer...
</summary>

| `min_samples_leaf` | out-of-bag | test |
| --: | --: | --: |
| 1 | 0.8427 | 0.8431 |
| **2** | **0.8616** | **0.8568** |
| 5 | 0.8600 | 0.8564 |
| 10 | 0.8590 | 0.8540 |
| 20 | 0.8555 | 0.8524 |
| 50 | 0.8495 | 0.8464 |

**The out-of-bag score picks 2, and so would the test set.** The two columns rank the six
settings identically, and the first did it without the test set being touched.

At `min_samples_leaf=2` the forest scores **0.8568**, ahead of the pruned tree's 0.8537, with
ROC AUC **0.9154** against 0.9067 — a wider margin than the accuracy shows.

Note how little separates leaf sizes 2 to 10. The gap is between 1 and everything else: what
mattered was stopping the trees at the last row, not the exact size.

</details>

## 7. When to use which

- **out-of-bag**: one fit, and only for bagged ensembles, since nothing else leaves rows out.
- **cross-validation**: five fits or ten, for any model, with a spread across folds.

For a forest with a grid to sweep, out-of-bag goes first and cross-validation confirms.

## Summary

- A bootstrap sample leaves about **37% of rows out**, and each row is predicted by the trees
  that missed it. `oob_score=True` computes that during the fit.
- On concrete it read **4.87 MPa** against the test set's 5.33; cross-validation, at five
  times the cost, read 5.28.
- A forest of **fully grown trees lost to one pruned tree** on census, 0.8431 against 0.8537.
- The out-of-bag score **chose `min_samples_leaf=2`**, as the test set would have, and the
  forest then won at **0.8568**, ROC AUC 0.9154.

Next, in `05_exercise_full_cycle.ipynb`: the whole workflow on your own, and a forest against
the tuned tree of the decision trees repository.